Prepare the regular-season dataset produced by `dataset_creation.ipynb`.

Use Spark `randomSplit([0.70, 0.15, 0.15], seed=42)` for training, validation, and test data. Fit categorical encoding and numerical imputation on training only, then apply them to all three sets. Write matching `data/random_full_traintest/` and `data/random_reduced_traintest/` splits for the classification notebooks.

The split is by pitch, not by game, pitcher, or season. It measures performance on randomly held-out pitches, and related pitches may appear in different sets. Random splitting is not stratified: inspect class counts after task-specific filters. The full and reduced versions retain the same pitch assignments.


In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    OneHotEncoder,
    StringIndexer,
)
from pyspark.sql import (
    SparkSession,
    functions as F,
)


In [ ]:
spark = (
    SparkSession.builder
    .appName("preprocessing")
    .master("local[2]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.parquet.columnarReaderBatchSize", "1024")
    .getOrCreate()
)


In [ ]:
spark.catalog.clearCache()
throws_df = spark.read.parquet("./data/full_dataset.parquet")


In [ ]:
throws_df.show(5)


In [ ]:
pitch_id = ["game_pk", "at_bat_number", "pitch_number"]

duplicate_pitches = (
    throws_df.groupBy(pitch_id)
    .count()
    .filter(F.col("count") > 1)
)

duplicate_count = duplicate_pitches.count()
print("Duplicated pitch identifiers:", duplicate_count)

throws_df.select(
    F.sum(F.when(~F.col("balls").between(0, 3), 1).otherwise(0)).alias("invalid_balls"),
    F.sum(F.when(~F.col("strikes").between(0, 2), 1).otherwise(0)).alias("invalid_strikes"),
    F.sum(F.when(F.col("inning") < 1, 1).otherwise(0)).alias("invalid_innings"),
    F.sum(F.when(F.col("release_speed") <= 0, 1).otherwise(0)).alias("invalid_release_speed")
).show()

valid_rows = (
    (F.col("balls").isNull() | F.col("balls").between(0, 3))
    & (F.col("strikes").isNull() | F.col("strikes").between(0, 2))
    & (F.col("inning").isNull() | (F.col("inning") >= 1))
    & (F.col("release_speed").isNull() | (F.col("release_speed") > 0))
)

throws_df = throws_df.filter(valid_rows)
throws_df = throws_df.dropDuplicates(pitch_id)
print("Rows after basic data cleaning:", throws_df.count())


In [ ]:
# Use the same feature groups as EDA.
deprecated = [
    "spin_dir",
    "spin_rate_deprecated",
    "break_angle_deprecated",
    "break_length_deprecated",
    "tfs_deprecated",
    "tfs_zulu_deprecated",
    "umpire",
    "sv_id",
    "age_pit_legacy",
    "age_bat_legacy",
]
fielders = [f"fielder_{n}" for n in range(1, 10)]
swing = [
    "bat_speed",
    "swing_length",
    "miss_distance",
    "hyper_speed",
    "attack_angle",
    "attack_direction",
    "swing_path_tilt",
    "intercept_ball_minus_batter_pos_x_inches",
    "intercept_ball_minus_batter_pos_y_inches",
]
post_scores = ["post_away_score", "post_home_score", "post_bat_score", "post_fld_score"]
postpitch_expectancy = ["delta_home_win_exp", "delta_run_exp", "delta_pitcher_run_exp"]
ids = [
    "game_pk",
    "game_date",
    "game_type",
    "at_bat_number",
    "pitch_number",
    "player_name",
    "home_team",
    "away_team",
    "pitcher",
    "batter",
]
result = ["type", "zone", "events", "des"]
batted = [
    "hit_location",
    "bb_type",
    "hc_x",
    "hc_y",
    "hit_distance_sc",
    "launch_speed",
    "launch_angle",
    "launch_speed_angle",
    "estimated_ba_using_speedangle",
    "estimated_woba_using_speedangle",
    "estimated_slg_using_speedangle",
    "babip_value",
    "iso_value",
    "woba_value",
    "woba_denom",
]

# Use pitch type as the target instead of pitch name.
prepared_df = (
    throws_df
    .withColumn(
        "zone_observed",
        F.col("plate_x").isNotNull()
        & F.col("plate_z").isNotNull()
        & F.col("sz_top").isNotNull()
        & F.col("sz_bot").isNotNull(),
    )
    .drop("pitch_name")
)

# Drop unused columns.
to_drop = [
    deprecated,
    fielders,
    swing,
    post_scores,
    postpitch_expectancy,
    ids,
    result,
    batted,
]
for x in to_drop:
    for y in x:
        prepared_df = prepared_df.drop(y)

prepared_df = prepared_df.drop(
    "pitcher_days_until_next_game",
    "batter_days_until_next_game",
    "home_score",
    "away_score",
    "home_score_diff",
)
print(len(prepared_df.columns))
print(len(throws_df.columns))


In [ ]:
prepared_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in prepared_df.columns
]).show()


In [ ]:
prepared_df.columns


In [ ]:
pitch = [
    "release_speed",
    "release_pos_x",
    "release_pos_y",
    "release_pos_z",
    "effective_speed",
    "release_spin_rate",
    "release_extension",
    "spin_axis",
    "arm_angle",
    "pfx_x",
    "pfx_z",
    "plate_x",
    "plate_z",
    "vx0",
    "vy0",
    "vz0",
    "ax",
    "ay",
    "az",
    "api_break_z_with_gravity",
    "api_break_x_arm",
    "api_break_x_batter_in",
]
bases = ["on_1b", "on_2b", "on_3b"]
alignments = ["if_fielding_alignment", "of_fielding_alignment"]
prepared_df = prepared_df.dropna(subset=["pitch_type"])

# Encode base occupancy instead of the player occupying the base.
for base in bases:
    prepared_df = prepared_df.withColumn(base, F.when(F.col(base).isNull(), 0).otherwise(1))
# Represent missing alignments with an explicit category.
for al in alignments:
    prepared_df = prepared_df.fillna("UNKNOWN", subset=al)


In [ ]:
# Numerical imputation is fitted on the training split below.


In [ ]:
# Pitch measurements used by the post-pitch classification experiment.


In [ ]:
categorical = [
    "stand",
    "p_throws",
    "inning_topbot",
    "balls",
    "strikes",
    "if_fielding_alignment",
    "of_fielding_alignment",
    "outs_when_up",
]

indexers = [
    StringIndexer(
        inputCol=col,
        outputCol=f"{col}_idx",
        handleInvalid="keep",
    )
    for col in categorical
]

encoder = OneHotEncoder(
    inputCols=[f"{col}_idx" for col in categorical],
    outputCols=[f"{col}_ohe" for col in categorical],
    handleInvalid="keep",
)


In [ ]:
# Split before fitting encoding or numerical imputation.
pipe = Pipeline(stages=indexers + [encoder])


def train_validation_test_split(df, pipeline):
    # Materialize cleaned data so all split branches use the same rows.
    df = df.cache()
    df.count()
    train, validation, test = df.randomSplit([0.70, 0.15, 0.15], seed=42)

    train = train.cache()
    validation = validation.cache()
    test = test.cache()
    for name, split in [("train", train), ("validation", validation), ("test", test)]:
        assert split.count() > 0, f"The {name} split is empty"

    pipe_model = pipeline.fit(train)
    train_encoded = pipe_model.transform(train)
    validation_encoded = pipe_model.transform(validation)
    test_encoded = pipe_model.transform(test)

    return train_encoded, validation_encoded, test_encoded

train, validation, test = train_validation_test_split(prepared_df, pipe)

numerical_features = list(dict.fromkeys(
    pitch + [
        "on_1b", "on_2b", "on_3b", "outs_when_up", "inning",
        "balls", "strikes", "sz_top", "sz_bot", "bat_score",
        "fld_score", "bat_score_diff", "home_win_exp", "bat_win_exp",
        "age_pit", "age_bat", "n_thruorder_pitcher",
        "n_priorpa_thisgame_player_at_bat",
        "pitcher_days_since_prev_game", "batter_days_since_prev_game"
    ]
))

quantiles = train.approxQuantile(numerical_features, [0.5], 0.01)
train_medians = {
    column: values[0]
    for column, values in zip(numerical_features, quantiles)
    if values
}
train = train.fillna(train_medians)
validation = validation.fillna(train_medians)
test = test.fillna(train_medians)

no_pitch_train = train.drop(*pitch)
no_pitch_validation = validation.drop(*pitch)
no_pitch_test = test.drop(*pitch)


In [ ]:
def report_dataset(df, name):
    print(name, "rows:", df.count())
    df.groupBy("pitch_type").count().orderBy(F.desc("count")).show(20)
    df.select([
        F.sum(F.col(column).isNull().cast("int")).alias(column)
        for column in numerical_features
    ]).show()

report_dataset(train, "train")
report_dataset(validation, "validation")
report_dataset(test, "test")


In [ ]:
train.write.mode("overwrite").parquet("./data/random_full_traintest/train")
validation.write.mode("overwrite").parquet("./data/random_full_traintest/validation")
test.write.mode("overwrite").parquet("./data/random_full_traintest/test")
no_pitch_train.write.mode("overwrite").parquet("./data/random_reduced_traintest/train")
no_pitch_validation.write.mode("overwrite").parquet("./data/random_reduced_traintest/validation")
no_pitch_test.write.mode("overwrite").parquet("./data/random_reduced_traintest/test")
spark.stop()
